# 02 -- Background rates and paper figures

Assumes notebook 01 has been run and the ingredients checked.  This notebook
exercises the top-level API and produces the summary figures.

The top-level call takes exactly what was asked for:

```python
nb.background_muons(
    detector    = nb.Cylinder(radius=7.5, half_length=10.5, depth=100.0),
    e_mu_min    = 10.0,             # GeV, at the detector
    e_nu_range  = (10.0, 1.0e6),    # GeV
    livetime_s  = nb.SEC_PER_YEAR,
)
```
and returns the arriving muon spectrum, the integrated count, and an error
budget.

In [ ]:
# --- make `nubkg` importable no matter where this notebook is opened from ----
# Looks for the nubkg package next to the notebook, then one level up, then
# falls back to whatever is already installed on sys.path.
import sys, pathlib

def _bootstrap():
    here = pathlib.Path.cwd()
    for cand in [here, *here.parents][:4]:
        if (cand / "nubkg" / "__init__.py").exists():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            return cand
    return None

_root = _bootstrap()
print("package root:", _root or "not found next to the notebook -- "
      "relying on an installed nubkg")

import numpy as np
import matplotlib.pyplot as plt
import nubkg as nb
from nubkg import plots as P

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.titlesize": 10, "figure.autolayout": True})

E_NU_RANGE = (10.0, 1.0e6)
LIVETIME = nb.SEC_PER_YEAR

flux = nb.TabulatedFlux.from_csv(
    nb.data_path("ic59_atmospheric_numu_APPROX.csv"),
    label="IC-59 unfolding (PLACEHOLDER) x Chirkin zenith shape",
    flux_unit="E2Phi", zenith_shape_from=nb.ChirkinAtmospheric(),
    shape_hemisphere="up")

cms = nb.Cylinder(radius=7.5, half_length=10.5, depth=100.0, name="CMS-like")
print(f"{cms.name}: V = {cms.volume_m3:.0f} m^3")

## Baseline result

In [ ]:
res = nb.background_muons(cms, e_mu_min=10.0, e_nu_range=E_NU_RANGE,
                          livetime_s=LIVETIME, flux=flux, hemisphere="up")
print(f"N(up-going, E_mu > 10 GeV, 1 yr) = {res['n_muons']:.1f} "
      f"+- {res['n_muons_err']:.1f}")
print()
for k, v in sorted(res["error_budget"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:15s} {v:7.2f}")

> **The dominant systematic is the low-energy end of the flux table.**
> The rate integrand peaks around $E_\nu \sim 100-300$ GeV (notebook 01,
> section 9), which is *below* where the IceCube unfolded points begin.  The
> digitised table therefore cannot determine the answer on its own: everything
> below its lowest point comes from whatever `outside=` policy is chosen.
> Compare the variants in the model-variation table further down -- blind
> log-log extrapolation of the table's steep high-energy slope roughly doubles
> the answer relative to splicing onto the analytic model.
>
> **Action item:** extend the table downward with the Frejus points from the
> same figure, or with Honda/MCEq, before quoting an absolute number.

## Figure 1: the arriving muon spectrum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
P.plot_arriving_spectrum(res, ax=axes[0])
P.plot_arriving_spectrum(res, ax=axes[1], cumulative=True)
axes[1].set_ylabel(r"N($>E_\mu$) per year")
plt.show()

## Figure 2: threshold scan

The number you cut on.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
_, scan = P.plot_threshold_scan(np.geomspace(10, 5e3, 14), cms, ax=ax,
                                livetime_s=LIVETIME, flux=flux,
                                e_nu_range=E_NU_RANGE)
plt.show()

print(f"{'E_thr [GeV]':>12} {'N / yr':>12} {'+-':>10}")
for t, n, e in zip(scan["thresholds"], scan["n"], scan["err"]):
    print(f"{t:12.0f} {n:12.3f} {e:10.3f}")

## Figure 3: error budget

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
P.plot_error_budget(res, ax=axes[0])
hi = nb.background_muons(cms, e_mu_min=1000., e_nu_range=E_NU_RANGE,
                         livetime_s=LIVETIME, flux=flux)
P.plot_error_budget(hi, ax=axes[1])
axes[1].set_title("error budget, E_thr = 1 TeV", fontsize=10)
plt.show()

## Model-variation table

Everything that is a *choice* rather than a measurement, varied one at a time.

In [ ]:
base = nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
                           livetime_s=LIVETIME, flux=flux,
                           uncertainty=False)["n_muons"]
variants = {
    "baseline (IC-59 table + qpm inelasticity)": base,
    "Chirkin analytic flux instead of table":
        nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
                            livetime_s=LIVETIME, flux=nb.ChirkinAtmospheric(),
                            uncertainty=False)["n_muons"],
    "table extrapolated below 350 GeV (BAD)":
        nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
            livetime_s=LIVETIME, uncertainty=False,
            flux=nb.TabulatedFlux.from_csv(
                nb.data_path("ic59_atmospheric_numu_APPROX.csv"),
                flux_unit="E2Phi", zenith_shape_from=nb.ChirkinAtmospheric(),
                outside="extrapolate"))["n_muons"],
    "no zenith shape on the table":
        nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
            livetime_s=LIVETIME, uncertainty=False,
            flux=nb.TabulatedFlux.from_csv(
                nb.data_path("ic59_atmospheric_numu_APPROX.csv"),
                flux_unit="E2Phi"))["n_muons"],
    "E_mu = E_nu (no inelasticity)":
        nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
                            livetime_s=LIVETIME, flux=flux,
                            inelasticity=nb.Inelasticity("none"),
                            uncertainty=False)["n_muons"],
    "flat dsigma/dy for both species":
        nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
                            livetime_s=LIVETIME, flux=flux,
                            inelasticity=nb.Inelasticity("flat"),
                            uncertainty=False)["n_muons"],
    "molasse instead of standard rock":
        nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
                            livetime_s=LIVETIME, flux=flux, rock=nb.MOLASSE,
                            uncertainty=False)["n_muons"],
    "no Earth absorption":
        nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
                            livetime_s=LIVETIME, flux=flux, absorb=False,
                            uncertainty=False)["n_muons"],
    "E_nu only above 100 GeV":
        nb.background_muons(cms, e_mu_min=10., e_nu_range=(100., 1e6),
                            livetime_s=LIVETIME, flux=flux,
                            uncertainty=False)["n_muons"],
}
print(f"{'variant':48s} {'N/yr':>8} {'ratio':>8}")
for k, v in variants.items():
    print(f"{k:48s} {v:8.2f} {v/base:8.2f}")

## Detector-size scan

The rate is proportional to projected area, not volume, so this is nearly a
straight line in $R$ with a mild upward curvature from the end caps.

In [ ]:
radii = np.linspace(2, 15, 10)
ns = [nb.background_muons(nb.Cylinder(radius=float(r), half_length=10.5,
                                      depth=100.),
                          e_mu_min=10., e_nu_range=E_NU_RANGE,
                          livetime_s=LIVETIME, flux=flux,
                          uncertainty=False)["n_muons"] for r in radii]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(radii, ns, "o-")
ax.set_xlabel("cylinder radius [m]"); ax.set_ylabel("background muons / yr")
ax.set_title("half-length fixed at 10.5 m, depth 100 m", fontsize=10)
ax.grid(alpha=.25); plt.show()

## Zenith distribution of the background

Relevant because the EarthShine signal points back at the Earth's centre within
$1.3^\circ\sqrt{{\rm TeV}/m_X}$ (1509.07525 Eq. 28), i.e. it is concentrated
at $\cos\theta \approx -1$, whereas this background is spread across the
whole up-going hemisphere and in fact peaks near the horizon.  **That is the
main handle for separating them.**

In [ ]:
res10 = nb.background_muons(cms, e_mu_min=10., e_nu_range=E_NU_RANGE,
                            livetime_s=LIVETIME, flux=flux, n_cos=60,
                            uncertainty=False)
cz = res10["cos_zenith"]
integ = np.trapezoid(res10["dPhi_dEmu"], res10["e_mu"], axis=1)
area = cms.mean_projected_area_m2(cz) * 1e4
d_rate = integ * area * 2 * np.pi * LIVETIME

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(cz, d_rate, lw=1.6)
ax.axvspan(-1.0, -0.999, color="crimson", alpha=.6,
           label=r"signal region ($m_X$ = 1 TeV)")
ax.set_xlabel(r"$\cos\theta_{\rm zenith}$")
ax.set_ylabel(r"$dN/d\cos\theta$ per year")
ax.legend(fontsize=8); ax.grid(alpha=.25); plt.show()

sig_frac = np.trapezoid(d_rate[cz < -0.999], cz[cz < -0.999]) if np.sum(cz < -0.999) > 1 else 0.
print(f"total up-going background: {np.trapezoid(d_rate, cz):.1f} / yr")
print("Fraction inside a 1.3 deg cone about straight-up is ~"
      f"{(1-np.cos(np.deg2rad(1.3)))/2:.2e} of the hemisphere solid angle,")
print("so the angular cut is worth roughly 4 orders of magnitude in background.")

## Metadata for the parquet footer

Matches the `b"earthshine"` footer convention so these results are
self-describing alongside the signal files.

In [ ]:
for k, v in res["metadata"].items():
    print(f"  {k:32s} {v}")